# CivicPulse Gauteng — Deprivation, Ward Incumbency & the Youth Participation Gap
**DIRISA Student Datathon Challenge 2026 · Teams Qualification**

**Core question:** To what extent do multi-dimensional deprivation (Stats SA Census 2022) and ward-level party incumbency (IEC 2016/2021) drive the youth voter participation gap across the City of Johannesburg and City of Tshwane?

| # | Section | What it answers |
|---|---------|-----------------|
| 1 | Problem understanding | What exactly are we measuring, at what unit, and is this regression or classification? |
| 2 | Setup & configuration | Libraries, paths, switches |
| 3 | Review of the original loader | Bugs that would silently corrupt the analysis, and how they are fixed |
| 4 | Data loading & understanding | Raw structure of every file, formats, duplicates, missingness |
| 5 | Harmonisation | Census → ward shares; IEC voting-district rows → ward results; 2016 → 2021 boundary reconciliation |
| 6 | Master table & data quality | One row per 2021 ward, coverage report |
| 7 | Statistical analysis | Descriptives, normality, correlations (FDR-corrected), multicollinearity, deprivation index validity |
| 8 | EDA | Distributions, relationships, metro and party contrasts |
| 9 | Hypothesis tests (H1, H2) | Standardised OLS with robust SEs, bootstrap, non-parametric tests |
| 10 | Predictive modelling — regression | 7 models under spatial cross-validation |
| 11 | Predictive modelling — classification | Low-turnout ward flagging |
| 12 | Model selection | Which model, and why — decided by the data with the one-standard-error rule |
| 13 | Interpretation | Coefficients, permutation importance, partial dependence |
| 14 | 2026 forecast & policy simulator | Ward-level forecast with intervals; what-if scenarios |
| 15 | Export & conclusions | Artefacts for the Streamlit app; data-driven verdicts on H1/H2 |

> **Run order:** top to bottom. If the real CSVs are not found, the notebook switches to a **synthetic demo dataset** so every cell can be tested. Synthetic numbers are meaningless — all findings must come from the real data.

---
## 1. Problem Understanding

### 1.1 Restating the problem
Low youth registration and turnout are often blamed on generic "apathy". The proposal argues a sharper claim: in Gauteng's metros, **local material conditions** (water, refuse, housing) and **the local political market** (how long one party has held the ward) shape whether people participate. Two testable hypotheses:

- **H1 — Deprivation friction.** Structural deprivation (refuse not collected, no piped water in the dwelling, informal dwellings) has a *larger* negative standardised effect on (youth) participation than income alone.
- **H2 — Incumbency demobilisation.** Where one party keeps the ward, turnout falls rather than votes moving to another party: voters *disengage* instead of *defect*.

### 1.2 Unit of analysis
The **ward**. Johannesburg had 135 wards and Tshwane 107 wards in the 2021 Local Government Elections, so **n ≈ 242**. That small sample drives almost every modelling decision below: simple, regularised models; honest cross-validation; confidence intervals rather than point estimates.

### 1.3 Operationalising the concepts

| Concept | Measured as | Source |
|---|---|---|
| Deprivation — water | % households without piped water inside the dwelling | Census 2022 |
| Deprivation — housing | % households in informal dwellings (backyard + non-backyard) | Census 2022 |
| Deprivation — refuse | % households *without* weekly municipal refuse removal | Census 2022 |
| Deprivation index | PCA first component and z-score average of the above | Derived |
| Youth weight | Share of adults aged 18–34 | Census 2022 age bands |
| Income (H1 comparison) | Median household income, if a ward-level file is available | Census / other |
| Incumbency | Same winning party in 2016 and 2021; winning margin; effective number of parties | IEC |
| Defection | Change in the 2016 winner's vote share, 2016 → 2021 | IEC |
| Participation | Turnout = votes cast / registered voters | IEC |
| Youth participation | Share of registered voters aged 18–29; youth registration rate vs Census population | IEC 2026 registration + Census |

### 1.4 A critical data-reality check
IEC's standard ward results report turnout for **all voters**, not by age. So **ward-level youth turnout is not directly observable** from the files in this project. We therefore use two targets, and are explicit about which one each result refers to:

- **T1 — Overall ward turnout (2021).** Fully observed. Primary target for the models.
- **T2 — Youth registration measures (2026).** The youth-specific gap we *can* observe: the share of registered voters aged 18–29 and the youth registration rate. Used for H1 when the registration file is present.

If your team finds an official age-by-ward turnout table, add it as a third target — the pipeline accepts any numeric target column.

### 1.5 Is this classification or prediction (regression)?
Both are *prediction*; the question is whether the target is continuous or categorical.

- **Regression is the primary framing.** Turnout is a continuous percentage and the hypotheses are about **effect sizes** ("greater negative effect"). Cutting turnout into classes throws away information and statistical power — costly with only ~242 wards.
- **Classification is a useful secondary framing** for operations: flag wards in the bottom turnout tercile ("demobilisation-risk wards") so campaigns can target them. Section 11 builds this and then tests, with data, whether a dedicated classifier beats simply ranking wards by the regression forecast.

### 1.6 Caveats stated up front
- **Ecological inference.** Ward-level associations do not tell us how *individual* young people behave.
- **Correlation, not causation.** Deprivation and incumbency are intertwined with history and geography; the policy simulator shows *model-implied* associations, not guaranteed effects.
- **Boundary change.** 2016 and 2021 wards are different polygons even where IDs match; Section 5 reconciles them through voting districts.

---
## 2. Setup & Configuration
### 2.1 Libraries
Standard scientific stack. `statsmodels` gives inference (robust standard errors, p-values); `scikit-learn` gives the predictive pipeline; XGBoost is used if installed, otherwise its slot is skipped automatically.

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.multitest import multipletests

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              RandomForestClassifier, GradientBoostingClassifier)
from sklearn.model_selection import KFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, roc_auc_score,
                             balanced_accuracy_score, f1_score, brier_score_loss)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import joblib

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

warnings.filterwarnings('ignore')
RNG = 42
np.random.seed(RNG)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')
RESULTS = {}   # every key finding is stored here and summarised in Section 15
print('XGBoost available:', HAS_XGB)

### 2.2 Mount Google Drive (Colab only)
Skipped automatically when running locally.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Running in Colab:', IN_COLAB)

### 2.3 Paths, files and switches
Edit this cell only. Optional layers (income, electricity, NEET, 2026 registration, ward centroids) are used if the file exists and skipped otherwise.

In [ ]:
import os
CANDIDATES = ['./data/', '/content/drive/MyDrive/CivicPulse_Gauteng_2026/data/', './src/data/', './']
BASE_DIR = next((d for d in CANDIDATES if os.path.exists(d) and any(f.endswith('.csv') for f in os.listdir(d))), './data/')
# BASE_DIR set above

CENSUS_FILES = {
    'water':  ['AccessToPipedWaters_Johannesburg.csv', 'AccessToPipedWaters_tshwane.csv'],
    'age':    ['AgeGroups_Johannesburg.csv',           'AgeGroups_Tshwane.csv'],
    'dwell':  ['dwelling_type_Johannesburg.csv',       'dwelling_type_tshwane.csv'],
    'refuse': ['RefuseDisposals_Johannesburg.csv',     'RefuseDisposals_tshwane.csv'],
    # Optional layers — add filenames when you have them:
    'electricity': [],
    'income': [],
    'neet': [],
}
ELECTION_FILES = {2016: '2016_GP_municipal_election.csv',
                  2021: '2021_GP_municipal_election.csv'}
REGISTRATION_FILE = 'IEC_registration_2026_by_age.csv'   # optional: ward x age-group x gender counts
CENTROIDS_FILE    = 'ward_centroids_2021.csv'            # optional: ward_id, lat, lon (better spatial CV)

FORCE_SYNTHETIC = False   # True = always use the demo data

required = [f for fs in CENSUS_FILES.values() for f in fs] + list(ELECTION_FILES.values())
missing = [f for f in required if not os.path.exists(os.path.join(BASE_DIR, f))]
USE_SYNTHETIC = FORCE_SYNTHETIC or len(missing) > 0
if USE_SYNTHETIC:
    print(f'SYNTHETIC DEMO MODE — {len(missing)} required file(s) not found in {BASE_DIR}')
    print('Results below test the code only. Do NOT report them.')
else:
    print('Real data found. Running on the project datasets.')

---
## 3. Review of the Original Loader — What Was Wrong and Why It Matters
The original `load_and_clean_all_layers()` runs without errors on many inputs, but several steps would **silently** produce a wrong master table. Each issue is fixed in Sections 4–6.

| # | Original code | Problem | Fix in this notebook |
|---|---|---|---|
| 1 | `str.contains('JHB\|TSH\|GT481\|GT484')` | Demarcation ward IDs are 8-digit numbers: Johannesburg wards start with **798**, Tshwane with **799**. `GT481`/`GT484` are Mogale City and Merafong, not the metros. On numeric IDs the filter keeps **zero rows** or the wrong municipalities. | Build full IDs, then filter on the `798`/`799` prefix; map short ward numbers (e.g. "Ward 12") using the municipality name or the filename. |
| 2 | Concatenate JHB + TSH census files, then use `ward` as key | If a file stores ward *numbers* (1–135), JHB Ward 1 and TSH Ward 1 collide. | Standardise IDs **per file** before concatenating. |
| 3 | `merge(election_16, election_21, on='ward_id')` | IEC files have one row per **party per voting district**. Merging two long tables is a many-to-many join: row counts explode (e.g. 30 rows × 30 rows per ward). | Aggregate IEC rows to one row per ward first. |
| 4 | `merge(water, age, ...)` | Census downloads are often long format (one row per category). Same explosion. | Pivot each census layer to one row per ward, convert counts to shares. |
| 5 | Matching 2016 and 2021 on ward ID | Wards were redrawn (JHB 130 → 135, TSH 105 → 107). Same ID ≠ same area. | Re-aggregate 2016 results onto 2021 wards through voting districts, and report coverage. |
| 6 | `fillna(median)` on the whole table | Imputes before cross-validation (information leaks from test wards) and also imputes targets. | Imputation happens inside the model pipeline, fitted on training folds only. Targets are never imputed. |
| 7 | Loop renames ward columns but then indexes `df['ward_id']` | Raises `KeyError` if a file's ward column has an unexpected name. | Robust column detection with a clear error message listing the columns found. |
| 8 | Registration data, income and NEET are in the proposal but not loaded | H1 cannot be tested on youth measures without them. | Optional loaders; the notebook reports clearly when a test cannot run. |

---
## 4. Data Loading & Understanding
### 4.1 Synthetic demo data (runs only if real files are missing)
Generates files in the **same raw formats** the real sources use (long-format census tables with "Ward 12"-style IDs; IEC voting-district × party rows with PR and Ward ballots; a non-metro municipality to test the filter). Relationships are invented purely to exercise the code.

In [ ]:
def make_synthetic_data(out_dir, seed=RNG):
    rng = np.random.default_rng(seed)
    os.makedirs(out_dir, exist_ok=True)
    softmax = lambda z: np.exp(z) / np.exp(z).sum()
    metros = {'Johannesburg': ('798', 135, 130, 'JHB'), 'Tshwane': ('799', 107, 105, 'TSH')}
    parties = ['AFRICAN NATIONAL CONGRESS', 'DEMOCRATIC ALLIANCE', 'ECONOMIC FREEDOM FIGHTERS',
               'ACTIONSA', 'INKATHA FREEDOM PARTY']
    age_bands = [f'{a} - {a+4}' for a in range(0, 85, 5)] + ['85+']
    age_w = np.array([9, 8.5, 8, 9, 10, 10, 9, 8, 7, 6, 5, 4, 3, 2.2, 1.5, 1, .6, .3])
    census = {k: {m: [] for m in metros} for k in ['water', 'age', 'dwell', 'refuse']}
    elec = {2016: [], 2021: []}
    reg = []
    vd_id = 32000000
    for metro, (pref, n21, n16, short) in metros.items():
        dep = np.convolve(rng.normal(0, 1, n21 + 8), np.ones(6) / 6, mode='same')[4:n21 + 4]
        dep = (dep - dep.mean()) / dep.std()
        mun = f'{short} - City of {metro}'
        for w in range(1, n21 + 1):
            d = dep[w - 1]; hh = int(rng.integers(6000, 16000)); ward = f'Ward {w}'
            def add(layer, cats, logits, n):
                for c, k in zip(cats, rng.multinomial(n, softmax(np.array(logits)))):
                    census[layer][metro].append({'Municipality': f'City of {metro}', 'Ward': ward,
                                                 'Category': c, 'Count': k})
            add('water', ['Piped (tap) water inside the dwelling', 'Piped (tap) water inside the yard',
                          'Piped (tap) water on community stand', 'No access to piped water'],
                [1.5 - 1.0*d, 0.5, -0.8 + 0.6*d, -2.2 + 0.8*d], hh)
            add('dwell', ['Formal dwelling/house', 'Flat or apartment', 'Informal dwelling/shack in backyard',
                          'Informal dwelling/shack not in backyard', 'Other'],
                [1.5 - 0.8*d, 0.2 - 0.4*d, -1.2 + 0.5*d, -1.6 + 1.0*d, -3], hh)
            add('refuse', ['Removed by local authority at least once a week', 'Removed by local authority less often',
                           'Communal refuse dump', 'Own refuse dump', 'No rubbish disposal'],
                [2.0 - 0.9*d, -1.5 + 0.3*d, -2 + 0.5*d, -2 + 0.7*d, -3 + 0.6*d], hh)
            tilt = 1 + 0.08*d + rng.normal(0, 0.05)
            aw = age_w.copy(); aw[3:7] *= tilt
            pop = hh * 3
            for c, k in zip(age_bands, rng.multinomial(pop, aw / aw.sum())):
                census['age'][metro].append({'Municipality': f'City of {metro}', 'Ward': ward,
                                             'Category': c, 'Count': k})
            # --- elections ---
            base = np.array([0.4 + 0.9*d, 0.3 - 0.9*d, -0.6 + 0.3*d, -1.0, -2.5]) + rng.normal(0, .35, 5)
            sh21 = softmax(base); sh16 = softmax(base + np.array([0.25, 0.1, -0.1, -9, 0]))
            anc_both = (sh21.argmax() == 0) and (sh16.argmax() == 0)
            t21 = np.clip(0.45 - 0.045*d - 0.02*anc_both + 0.02*(metro == 'Tshwane') + rng.normal(0, .035), .15, .8)
            t16 = np.clip(t21 + 0.08 + rng.normal(0, .03), .15, .85)
            registered = int(pop * rng.uniform(.38, .5))
            n_vd = int(rng.integers(6, 11)); vd_share = rng.dirichlet(np.ones(n_vd) * 3)
            w16 = int(np.ceil(w * n16 / n21))
            for j in range(n_vd):
                vd_id += 1
                for year, t, sh, wid in [(2021, t21, sh21, pref + str(w).zfill(5)),
                                         (2016, t16, sh16, pref + str(w16).zfill(5))]:
                    if year == 2016 and rng.random() < 0.10:   # ~10% of VDs are new in 2021
                        continue
                    r = int(registered * vd_share[j] * (0.93 if year == 2016 else 1))
                    cast = rng.binomial(r, t); spoilt = rng.binomial(cast, 0.012)
                    votes = rng.multinomial(cast - spoilt, sh)
                    for ballot, jitter in [('Ward', 0), ('PR', 1)]:
                        for p, v in zip(parties, votes):
                            if year == 2016 and p == 'ACTIONSA':
                                continue
                            elec[year].append({'Province': 'Gauteng', 'Municipality': mun, 'Ward': wid,
                                               'VotingDistrict': vd_id, 'BallotType': ballot, 'PartyName': p,
                                               'RegisteredVoters': r, 'SpoiltVotes': spoilt,
                                               'TotalValidVotes': max(0, v + jitter * int(rng.integers(-3, 4)))})
            # --- 2026 registration by age ---
            y_rate = np.clip(0.46 - 0.07*d + rng.normal(0, .04), .1, .9)
            y_pop = pop * 0.24
            for g in ['Male', 'Female']:
                for band, n in [('18-19', y_pop*.17*y_rate), ('20-29', y_pop*.83*y_rate), ('30-39', pop*.17*.72),
                                ('40-49', pop*.12*.8), ('50-59', pop*.08*.85), ('60-69', pop*.05*.85),
                                ('70-79', pop*.025*.8), ('80+', pop*.01*.7)]:
                    reg.append({'Ward': pref + str(w).zfill(5), 'AgeGroup': band, 'Gender': g,
                                'RegisteredVoters': int(n / 2)})
    # a non-metro municipality to prove the filter works
    for year in (2016, 2021):
        elec[year].append({'Province': 'Gauteng', 'Municipality': 'GT481 - Mogale City', 'Ward': '74801001',
                           'VotingDistrict': 1, 'BallotType': 'Ward', 'PartyName': parties[0],
                           'RegisteredVoters': 5000, 'SpoiltVotes': 20, 'TotalValidVotes': 2000})
    for layer, fnames in CENSUS_FILES.items():
        for fname in fnames:
            metro = 'Johannesburg' if 'johannesburg' in fname.lower() else 'Tshwane'
            df = pd.DataFrame(census[layer][metro])
            df.loc[df.sample(frac=0.01, random_state=seed).index, 'Count'] = np.nan
            df.to_csv(os.path.join(out_dir, fname), index=False)
    for year, fname in ELECTION_FILES.items():
        pd.DataFrame(elec[year]).to_csv(os.path.join(out_dir, fname), index=False)
    pd.DataFrame(reg).to_csv(os.path.join(out_dir, REGISTRATION_FILE), index=False)

if USE_SYNTHETIC:
    BASE_DIR = '/tmp/civicpulse_synthetic/'
    make_synthetic_data(BASE_DIR)
    print('Synthetic files written to', BASE_DIR, '->', sorted(os.listdir(BASE_DIR)))

### 4.2 Generic helpers: column normalisation and ward-ID standardisation
Every source names its columns differently ("Ward", "WARD_ID", "Electoral Ward"…). These helpers make each table speak the same language:

- `norm_cols` lower-cases names and replaces symbols with `_`.
- `standardize_ward_id` produces the official 8-digit Municipal Demarcation Board ward ID (`798xxxxx` = Johannesburg, `799xxxxx` = Tshwane) and keeps only metro rows. Short ward numbers are expanded using the municipality column or, failing that, the filename.

In [ ]:
METRO_PREFIX = {'Johannesburg': '798', 'Tshwane': '799'}
PREFIX_METRO = {v: k for k, v in METRO_PREFIX.items()}
WARD_CANDIDATES = ['ward_id', 'wardid', 'ward', 'ward_number', 'ward_no', 'electoral_ward', 'ward_code']
MUNI_CANDIDATES = ['municipality', 'municipality_name', 'municipalityname', 'municname', 'mun_name',
                   'metro', 'local_municipality', 'munic']

def norm_name(s):
    return re.sub(r'[^a-z0-9]+', '_', str(s).strip().lower()).strip('_')

def norm_cols(df):
    df = df.copy()
    df.columns = [norm_name(c) for c in df.columns]
    return df

def find_col(df, candidates, contains=False):
    for c in candidates:
        if c in df.columns:
            return c
    if contains:
        for c in df.columns:
            if any(k in c for k in candidates):
                return c
    return None

def read_csv_safe(path):
    for enc in ('utf-8-sig', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'Could not decode {path}')

def metro_from_text(txt):
    t = str(txt).lower()
    if 'johannesburg' in t or 'jhb' in t: return 'Johannesburg'
    if 'tshwane' in t or 'tsh' in t:      return 'Tshwane'
    return None

def standardize_ward_id(df, metro_hint=None, source=''):
    df = norm_cols(df)
    wcol = None
    if 'geolevelvaluedesc' in df.columns:            # Stats SA Census 2022 portal export
        levels = df['geoleveldesc'].astype(str).str.lower().unique() if 'geoleveldesc' in df else []
        if len(levels) and not any('ward' in l for l in levels):
            raise ValueError(f'[{source}] is exported at geography level {list(levels)}, not Ward. '
                             'Re-export this indicator at Ward level for each metro.')
        wcol = 'geolevelvaluedesc'
    wcol = wcol or find_col(df, WARD_CANDIDATES) or next((c for c in df.columns if 'ward' in c), None)
    if wcol is None:
        raise KeyError(f'[{source}] no ward column found. Columns: {list(df.columns)}')
    digits = df[wcol].astype(str).str.replace(r'\.0$', '', regex=True).str.extract(r'(\d+)')[0]
    mcol = find_col(df, MUNI_CANDIDATES, contains=True)
    munis = df[mcol].astype(str).tolist() if mcol else [None] * len(df)
    ids = []
    for dg, mu in zip(digits.tolist(), munis):
        if not isinstance(dg, str):
            ids.append(np.nan); continue
        if len(dg) == 8:
            ids.append(dg); continue
        metro = metro_from_text(mu) if mu else None
        metro = metro or metro_hint
        ids.append(METRO_PREFIX[metro] + dg.zfill(5) if (metro and len(dg) <= 3) else np.nan)
    df['ward_id'] = pd.Series(ids, index=df.index, dtype=object)
    if wcol != 'ward_id':
        df = df.drop(columns=[wcol])
    df['metro'] = df['ward_id'].astype(str).str[:3].map(PREFIX_METRO)
    n_in = len(df)
    df = df[df['metro'].notna()].copy()
    print(f'  [{source}] ward column="{wcol}", municipality column="{mcol}", kept {len(df):,}/{n_in:,} rows '
          f'({df.ward_id.nunique()} wards)')
    return df

### 4.3 Load every raw file and look at it before transforming anything
Data understanding starts with the raw shape: how many rows per ward (1 = wide format, many = long format), which columns exist, and where values are missing. The **rows-per-ward** column is the direct evidence for issues #3 and #4 in Section 3.

In [ ]:
raw_census = {}
for layer, files in CENSUS_FILES.items():
    parts = []
    for f in files:
        path = os.path.join(BASE_DIR, f)
        if not os.path.exists(path):
            print(f'  [{layer}] {f} not found — skipped'); continue
        parts.append(standardize_ward_id(read_csv_safe(path), metro_from_text(f), source=f))
    if parts:
        raw_census[layer] = pd.concat(parts, ignore_index=True)

raw_elec = {}
for year, f in ELECTION_FILES.items():
    raw_elec[year] = standardize_ward_id(read_csv_safe(os.path.join(BASE_DIR, f)), None, source=f)

def profile(name, df):
    return {'table': name, 'rows': len(df), 'cols': df.shape[1], 'wards': df['ward_id'].nunique(),
            'rows_per_ward': round(len(df) / max(df['ward_id'].nunique(), 1), 1),
            'missing_cells_%': round(df.isna().mean().mean() * 100, 2),
            'columns': ', '.join(df.columns[:12])}

prof = pd.DataFrame([profile(k, v) for k, v in raw_census.items()] +
                    [profile(f'election_{k}', v) for k, v in raw_elec.items()])
display(prof)

In [ ]:
# Peek at the first rows of each raw table to confirm category names and value columns
for k, v in {**raw_census, **{f'election_{y}': d for y, d in raw_elec.items()}}.items():
    print(f'\n--- {k} ---')
    display(v.head(4))

**What to check here (real data):** that each census table has either one row per ward (wide) or a category column plus a count column (long); that IEC tables have a party column, a votes column and a registered-voters column; and that both metros appear with plausible ward counts (≈135 and ≈107 for 2021).

---
## 5. Harmonisation
### 5.1 Census layers → one row per ward, counts → shares
Each census layer is pivoted to **ward × category counts** (if long) and converted to **shares of households/persons**, so wards of different size are comparable. Columns whose names look like totals are dropped so they don't double the denominator.

In [ ]:
DROP_PATTERN = r'(^|_)(year|code|lat|lon|latitude|longitude|shape|objectid|fid|province|district)($|_)'
VALUE_CANDIDATES = ['count', 'value', 'total', 'population', 'households', 'persons', 'number', 'freq']

def census_counts_wide(df, layer):
    if 'label' in df.columns and 'counts' in df.columns:     # Stats SA portal format
        cnt = pd.to_numeric(df['counts'], errors='coerce')
        if (cnt.fillna(0) == 0).all() and 'countsmales' in df:
            cnt = pd.to_numeric(df['countsmales'], errors='coerce') + pd.to_numeric(df['countsfemales'], errors='coerce')
        if (cnt.fillna(0) == 0).all() and 'countspercentage' in df:
            cnt = pd.to_numeric(df['countspercentage'], errors='coerce')
        df = df[['ward_id', 'metro', 'label']].assign(value=cnt)
    cand = [c for c in df.columns if c not in ('ward_id', 'metro') and not re.search(DROP_PATTERN, c)]
    num = [c for c in cand if pd.api.types.is_numeric_dtype(df[c])]
    if df['ward_id'].duplicated().any():                      # long format -> pivot
        txt = [c for c in cand if c not in num and not re.search(r'munic|metro|name|ward', c)]
        val = find_col(df[num], VALUE_CANDIDATES, contains=True) if num else None
        val = val or (num[-1] if num else None)
        if not txt or val is None:
            raise ValueError(f'[{layer}] long format but no category/value column found: {cand}')
        wide = df.pivot_table(index='ward_id', columns=txt[0], values=val, aggfunc='sum')
        print(f'  {layer}: long -> wide using category="{txt[0]}", value="{val}"')
    else:
        wide = df.set_index('ward_id')[num]
        print(f'  {layer}: already wide ({len(num)} numeric columns)')
    wide.columns = [f'{layer}__{norm_name(c)}' for c in wide.columns]
    wide = wide.drop(columns=[c for c in wide.columns if re.search(r'__(grand_)?total$', c)])
    return wide

census_counts = {layer: census_counts_wide(df, layer) for layer, df in raw_census.items()}
SHARE_LAYERS = [l for l in census_counts if l not in ('income', 'neet')]
census_shares = {l: census_counts[l].div(census_counts[l].sum(axis=1, min_count=1), axis=0) for l in SHARE_LAYERS}
for l, s in census_shares.items():
    print(f'{l:12s} wards={len(s):4d} categories={s.shape[1]}')

### 5.2 Deprivation features from category keywords
Features are defined by keyword rules on category names. The cell **prints exactly which categories were combined** for each feature — verify this against the real Census 2022 labels and edit `FEATURE_RULES` if a label differs. `invert=True` means "1 − share", e.g. deprivation = share **without** weekly refuse removal.

In [ ]:
#  feature: (layer, include-patterns, exclude-patterns, invert)
FEATURE_RULES = {
    'pct_no_piped_water':         ('water',       [r'no access', r'no piped'],            [], False),
    'pct_water_outside_dwelling': ('water',       [r'inside (the )?dwelling'],            [], True),
    'pct_informal_dwelling':      ('dwell',       [r'informal', r'shack'],                [], False),
    'pct_refuse_not_weekly':      ('refuse',      [r'at least once a week', r'weekly'],   [], True),
    'pct_no_electricity':         ('electricity', [r'\bnone\b', r'no electricity'],       [], False),
}
DIRECT_RULES = {'median_income': ('income', r'median'), 'youth_neet_rate': ('neet', r'neet|rate')}

all_census_wards = sorted(set().union(*[set(s.index) for s in census_shares.values()]))
features = pd.DataFrame(index=pd.Index(all_census_wards, name='ward_id'))
feature_log = []
for fname, (layer, inc, exc, invert) in FEATURE_RULES.items():
    if layer not in census_shares:
        feature_log.append((fname, 'SKIPPED — layer not loaded')); continue
    sh = census_shares[layer]
    label = lambda c: c.split('__', 1)[1].replace('_', ' ')
    cols = [c for c in sh.columns if any(re.search(p, label(c)) for p in inc)
            and not any(re.search(p, label(c)) for p in exc)]
    if not cols:
        feature_log.append((fname, f'NOT BUILT — no category matched {inc}')); continue
    v = sh[cols].sum(axis=1)
    features[fname] = (1 - v) if invert else v
    feature_log.append((fname, ('1 - ' if invert else '') + ' + '.join(label(c) for c in cols)))
for fname, (layer, pat) in DIRECT_RULES.items():
    if layer in census_counts:
        cols = [c for c in census_counts[layer].columns if re.search(pat, c.split('__', 1)[1])]
        if cols:
            features[fname] = census_counts[layer][cols[0]]
            feature_log.append((fname, f'direct column {cols[0]}')); continue
    feature_log.append((fname, 'SKIPPED — no file/column'))
display(pd.DataFrame(feature_log, columns=['feature', 'built from']))

### 5.3 Age structure → youth share and the 2026 youth cohort
Age bands are parsed from the category names (e.g. "15 - 19", "85+"). Population inside an age range is computed assuming ages are spread evenly within a band (so 18–19 = 2/5 of the 15–19 band).

- `youth_share_adults` = persons aged 18–34 / persons 18+.
- `pop_18_29_2026` = persons who will be 18–29 on election day 2026. Census night was October 2022, about four years earlier, so this is the Census population aged **14–25**. It is the denominator for the youth registration rate.

In [ ]:
def parse_band(name):
    nums = [int(x) for x in re.findall(r'\d+', name)]
    if len(nums) >= 2 and nums[1] >= nums[0]:
        return nums[0], nums[1]
    if len(nums) == 1:   # open-ended top band such as "85+" / "80 and over"
        if re.search(r'over|plus|above|older', name) or nums[0] >= 80:
            return nums[0], 110
        return nums[0], nums[0]
    return None

def pop_between(counts, bands, a, b):
    total = pd.Series(0.0, index=counts.index)
    for c, (lo, hi) in bands.items():
        overlap = max(0, min(hi, b) - max(lo, a) + 1)
        if overlap:
            total = total + counts[c].fillna(0) * overlap / (hi - lo + 1)
    return total

if 'age' in census_counts:
    ac = census_counts['age']
    bands = {c: parse_band(c.split('__', 1)[1]) for c in ac.columns}
    bands = {c: b for c, b in bands.items() if b}
    print('Parsed age bands:', sorted(bands.values())[:20])
    adults = pop_between(ac, bands, 18, 110)
    features['youth_share_adults'] = pop_between(ac, bands, 18, 34) / adults
    features['pop_adults'] = adults
    features['pop_18_29_2026'] = pop_between(ac, bands, 14, 25)
display(features.describe().T)

### 5.4 IEC results: voting-district × party rows → ward results
Steps: keep only the **Ward ballot** (drop PR and DC ballots, which would double-count votes); de-duplicate registered voters and spoilt ballots per voting district (they repeat on every party row); sum party votes; then compute per ward:

- `turnout` = (valid + spoilt) / registered (or votes cast / registered if that column exists)
- `winner`, `winner_share`, `margin` (winner − runner-up share)
- `enp` = effective number of parties, 1 / Σ share² (1 = one-party dominance)
- vote shares of ANC, DA, EFF

In [ ]:
PARTY_ABBR = {'african national congress': 'ANC', 'democratic alliance': 'DA', 'economic freedom fighters': 'EFF',
              'actionsa': 'ACTIONSA', 'inkatha freedom party': 'IFP', 'freedom front plus': 'FF+',
              'vryheidsfront plus': 'FF+', 'patriotic alliance': 'PA', 'independent': 'IND'}

def party_abbr(p):
    p = str(p).strip().lower()
    for k, v in PARTY_ABBR.items():
        if k in p:
            return v
    return p.upper()[:20]

def parse_iec(df, year):
    c = {k: find_col(df, v) for k, v in {
        'party': ['partyname', 'party_name', 'party', 'party_abbreviation'],
        'votes': ['totalvalidvotes', 'total_valid_votes', 'validvotes', 'valid_votes', 'votes', 'party_votes'],
        'reg':   ['registeredvoters', 'registered_voters', 'registered', 'reg_voters'],
        'spoilt': ['spoiltvotes', 'spoilt_votes', 'spoilt', 'spoilt_ballots'],
        'cast':  ['totalvotescast', 'total_votes_cast', 'votes_cast'],
        'vd':    ['votingdistrict', 'voting_district', 'vd', 'vd_number', 'vdnumber'],
        'ballot': ['ballottype', 'ballot_type', 'ballot']}.items()}
    print(f'{year}: detected columns {c}')
    if c['ballot']:
        b = df[c['ballot']].astype(str).str.lower()
        df = df[b.str.contains('ward') | b.eq('w')].copy()
        print(f'  kept Ward ballot only -> {len(df):,} rows')
    if c['party'] is None or c['votes'] is None or c['reg'] is None:
        raise KeyError(f'{year}: need party, votes and registered columns. Found {list(df.columns)}')
    df['vd'] = df[c['vd']].astype(str) if c['vd'] else df['ward_id']
    meta_cols = {'registered': c['reg'], 'spoilt': c['spoilt'], 'cast': c['cast']}
    meta = df.groupby(['ward_id', 'vd']).agg(**{k: (v, 'max') for k, v in meta_cols.items() if v}).reset_index()
    for k in meta_cols:
        if k not in meta: meta[k] = np.nan
    votes = df.assign(party=df[c['party']].map(party_abbr)).groupby(['ward_id', 'vd', 'party'])[c['votes']] \
              .sum().rename('votes').reset_index()
    return meta, votes, c['vd'] is not None

def summarise_results(meta, votes, key='ward_id'):
    reg = meta.groupby(key)[['registered', 'spoilt', 'cast']].sum(min_count=1)
    pv = votes.groupby([key, 'party'])['votes'].sum().unstack(fill_value=0)
    valid = pv.sum(axis=1)
    shares = pv.div(valid, axis=0)
    out = pd.DataFrame(index=reg.index)
    out['registered'] = reg['registered']
    out['valid'] = valid
    out['spoilt_rate'] = reg['spoilt'] / (valid + reg['spoilt'])
    cast = reg['cast'] if reg['cast'].notna().any() else valid + reg['spoilt'].fillna(0)
    out['turnout'] = cast / reg['registered']
    srt = np.sort(shares.values, axis=1)[:, ::-1]
    out['winner'] = shares.idxmax(axis=1)
    out['winner_share'] = srt[:, 0]
    out['margin'] = srt[:, 0] - srt[:, 1]
    out['enp'] = 1 / (shares ** 2).sum(axis=1)
    for p in ['ANC', 'DA', 'EFF']:
        out[f'share_{p.lower()}'] = shares[p] if p in shares else 0.0
    return out, shares

iec = {y: parse_iec(d, y) for y, d in raw_elec.items()}
res21, shares21 = summarise_results(iec[2021][0], iec[2021][1])
print(f'\n2021: {len(res21)} wards | turnout mean {res21.turnout.mean():.3f}')
display(res21.head())

### 5.5 Reconciling 2016 results onto 2021 wards (boundary change)
Voting districts are much smaller than wards and many keep their number across elections. We take each **2016 voting district**, look up which **2021 ward** that voting district belongs to, and re-aggregate 2016 results onto 2021 wards. The share of 2016 registered voters that could be mapped is reported — if it is low, H2 results should be read with caution. If the IEC files have no voting-district column, the notebook falls back to naive ID matching and warns you.

In [ ]:
meta16, votes16, has_vd16 = iec[2016]
meta21, votes21, has_vd21 = iec[2021]
if has_vd16 and has_vd21:
    vd_to_ward21 = meta21.drop_duplicates('vd').set_index('vd')['ward_id']
    meta16 = meta16.assign(ward21=meta16['vd'].map(vd_to_ward21))
    votes16 = votes16.assign(ward21=votes16['vd'].map(vd_to_ward21))
    coverage = meta16.loc[meta16.ward21.notna(), 'registered'].sum() / meta16['registered'].sum()
    res16, shares16 = summarise_results(meta16.dropna(subset=['ward21']), votes16.dropna(subset=['ward21']), key='ward21')
    res16.index.name = 'ward_id'; shares16.index.name = 'ward_id'
    # wards where the mapped 2016 electorate is tiny are unreliable
    reg21 = res21['registered']
    res16['mapped_reg_ratio'] = res16['registered'] / reg21.reindex(res16.index)
    RECON_METHOD = 'voting-district crosswalk'
else:
    coverage = np.nan
    res16, shares16 = summarise_results(meta16, votes16)
    res16['mapped_reg_ratio'] = np.nan
    RECON_METHOD = 'naive ward-ID match (WARNING: boundaries differ)'
print(f'Method: {RECON_METHOD}')
print(f'Share of 2016 registered voters mapped onto 2021 wards: {coverage:.1%}')
print(f'2021 wards with 2016 data: {res16.index.isin(res21.index).sum()} / {len(res21)}')
RESULTS['recon_method'] = RECON_METHOD; RESULTS['recon_coverage'] = float(coverage) if coverage == coverage else None

### 5.6 Optional: 2026 registration by age (youth-specific target T2)
Produces `youth_reg_share_2026` (18–29 year-olds as a share of all registered voters) and `youth_reg_rate_2026` (registered 18–29 / Census cohort that will be 18–29 in 2026). The rate is an approximation: migration since 2022 and Census undercount both add noise, so values are clipped to [0, 1.2] and flagged above 1.

In [ ]:
registration = None
reg_path = os.path.join(BASE_DIR, REGISTRATION_FILE)
if os.path.exists(reg_path):
    r = standardize_ward_id(read_csv_safe(reg_path), None, source=REGISTRATION_FILE)
    age_col = find_col(r, ['agegroup', 'age_group', 'age_band', 'age'], contains=True)
    val_col = find_col(r, ['registeredvoters', 'registered_voters', 'registered', 'count', 'total'], contains=True)
    wide = r.pivot_table(index='ward_id', columns=age_col, values=val_col, aggfunc='sum')
    rb = {c: parse_band(norm_name(c)) for c in wide.columns}
    youth_cols = [c for c, b in rb.items() if b and b[0] >= 18 and b[1] <= 29]
    print('Youth (18-29) registration columns:', youth_cols)
    registration = pd.DataFrame({'registered_2026': wide.sum(axis=1),
                                 'youth_registered_2026': wide[youth_cols].sum(axis=1)})
    registration['youth_reg_share_2026'] = registration.youth_registered_2026 / registration.registered_2026
    if 'pop_18_29_2026' in features:
        rate = registration.youth_registered_2026 / features['pop_18_29_2026'].reindex(registration.index)
        print(f'Wards with rate > 1 (denominator issues): {(rate > 1).sum()}')
        registration['youth_reg_rate_2026'] = rate.clip(0, 1.2)
    display(registration.describe().T)
else:
    print('No registration file — youth target T2 unavailable; analysis uses overall turnout (T1).')

---
## 6. Master Table & Data Quality
One row per **2021 ward** in the two metros. 2021 results define the universe; census, reconciled 2016 results and registration are left-joined, so missing pieces show up as NaN rather than silently dropping wards (the original inner joins hid losses).

Derived H2 variables:
- `retained` = same winning party in 2016 and 2021 (proxy for prolonged tenure — two consecutive terms at minimum).
- `d_turnout` = turnout 2021 − turnout 2016 (**disengagement** signal).
- `d_incumbent_share` = 2021 share − 2016 share of the party that won in 2016 (**defection** signal).

In [ ]:
r21 = res21.add_suffix('_2021')
r16 = res16.add_suffix('_2016')
master = r21.join(r16, how='left').join(features, how='left')
if registration is not None:
    master = master.join(registration, how='left')
master.index.name = 'ward_id'
master = master.reset_index()
master['metro'] = master['ward_id'].str[:3].map(PREFIX_METRO)
master['ward_num'] = master['ward_id'].str[-5:].astype(int)

master['retained'] = (master['winner_2016'] == master['winner_2021']).astype(float)
master.loc[master['winner_2016'].isna(), 'retained'] = np.nan
master['anc_retained'] = ((master['winner_2016'] == 'ANC') & (master['winner_2021'] == 'ANC')).astype(float)
master['d_turnout'] = master['turnout_2021'] - master['turnout_2016']
inc_2021 = [shares21.loc[w, p] if (w in shares21.index and isinstance(p, str) and p in shares21.columns) else np.nan
            for w, p in zip(master.ward_id, master.winner_2016)]
master['d_incumbent_share'] = np.array(inc_2021, dtype=float) - master['winner_share_2016']
for y in (2016, 2021):
    master[f'winner_grp_{y}'] = master[f'winner_{y}'].where(master[f'winner_{y}'].isin(['ANC', 'DA']), 'OTHER')

# Deprivation index: one water indicator only, to avoid double-weighting water
INDEX_COMPONENTS = [c for c in ['pct_water_outside_dwelling', 'pct_informal_dwelling',
                                'pct_refuse_not_weekly', 'pct_no_electricity'] if c in master]
Z = master[INDEX_COMPONENTS].apply(lambda s: (s - s.mean()) / s.std())
master['dep_index_z'] = Z.mean(axis=1)
pca_mask = Z.notna().all(axis=1)
pca = PCA(n_components=1).fit(Z[pca_mask])
sign = np.sign(pca.components_[0].sum())            # orient so higher = more deprived
master.loc[pca_mask, 'dep_index_pca'] = sign * pca.transform(Z[pca_mask])[:, 0]
print('Index components:', INDEX_COMPONENTS)
print(f'PCA PC1 explains {pca.explained_variance_ratio_[0]:.1%} of variance; loadings:',
      dict(zip(INDEX_COMPONENTS, np.round(sign * pca.components_[0], 3))))
RESULTS['pca_var_explained'] = float(pca.explained_variance_ratio_[0])
print('Master shape:', master.shape)

In [ ]:
# Data-quality report
EXPECTED = {'Johannesburg': 135, 'Tshwane': 107}
cov = master.groupby('metro').agg(wards=('ward_id', 'nunique'),
                                  with_census=('pct_informal_dwelling', lambda s: s.notna().sum()),
                                  with_2016=('turnout_2016', lambda s: s.notna().sum()))
cov['expected_2021'] = cov.index.map(EXPECTED)
display(cov)
miss = master.isna().mean().mul(100).round(1)
display(miss[miss > 0].sort_values(ascending=False).to_frame('missing_%'))
print('Duplicate ward_ids:', master.ward_id.duplicated().sum())
print('Turnout outside [0,1]:', ((master.turnout_2021 < 0) | (master.turnout_2021 > 1)).sum())

**Reading the report:** ward counts should match 135 / 107. Wards missing census data usually mean an ID mismatch (check the printed keep-rates in 4.3). Wards missing 2016 data are voting districts new in 2021 — they are kept, and handled by imputation inside the model pipeline plus a missing-indicator feature.

---
## 7. Statistical Analysis
### 7.1 Variable groups used from here on

In [ ]:
DEPRIVATION = [c for c in ['pct_no_piped_water', 'pct_water_outside_dwelling', 'pct_informal_dwelling',
                           'pct_refuse_not_weekly', 'pct_no_electricity', 'youth_neet_rate'] if c in master]
SOCIO = [c for c in ['median_income', 'youth_share_adults'] if c in master and master[c].notna().any()]
POLITICAL_2021 = ['margin_2021', 'enp_2021', 'share_anc_2021', 'share_da_2021', 'share_eff_2021', 'spoilt_rate_2021']
TARGETS = [t for t in ['turnout_2021', 'youth_reg_share_2026', 'youth_reg_rate_2026'] if t in master]
print('Deprivation:', DEPRIVATION); print('Socio:', SOCIO); print('Targets:', TARGETS)

### 7.2 Descriptive statistics
Mean, spread, skewness and missingness for every analytical variable. Skewness above |1| signals that means can mislead and that rank-based (Spearman, Mann-Whitney) methods are safer.

In [ ]:
desc_cols = TARGETS + ['turnout_2016', 'd_turnout', 'd_incumbent_share'] + DEPRIVATION + SOCIO + \
            ['dep_index_z', 'dep_index_pca'] + POLITICAL_2021
desc_cols = [c for c in dict.fromkeys(desc_cols) if c in master]
desc = master[desc_cols].describe().T
desc['skew'] = master[desc_cols].skew()
desc['missing_%'] = master[desc_cols].isna().mean() * 100
display(desc.round(3))
display(master.groupby('metro')[TARGETS + ['dep_index_z']].agg(['mean', 'median', 'std']).round(3))

### 7.3 Normality and group differences
- **Shapiro–Wilk** tests whether turnout is normally distributed (relevant for choosing parametric vs rank tests; OLS itself only needs roughly normal *residuals*).
- **Mann–Whitney U**: do Johannesburg and Tshwane differ in turnout? (Justifies a metro control.)
- **Kruskal–Wallis**: does turnout differ by the 2021 winning party group?

In [ ]:
t = master['turnout_2021'].dropna()
sw = stats.shapiro(t)
print(f'Shapiro-Wilk turnout_2021: W={sw.statistic:.3f}, p={sw.pvalue:.4f} ->',
      'not normal (use rank tests)' if sw.pvalue < 0.05 else 'normality not rejected')
g = [d['turnout_2021'].dropna() for _, d in master.groupby('metro')]
mw = stats.mannwhitneyu(*g)
print(f'Mann-Whitney JHB vs TSH turnout: U={mw.statistic:.0f}, p={mw.pvalue:.4f}')
groups = [d['turnout_2021'].dropna() for _, d in master.groupby('winner_grp_2021') if len(d) >= 5]
kw = stats.kruskal(*groups)
print(f'Kruskal-Wallis turnout by winner group: H={kw.statistic:.2f}, p={kw.pvalue:.4f}')
RESULTS['metro_diff_p'] = float(mw.pvalue)

### 7.4 Correlations with multiple-testing control
Spearman correlations (robust to skew and outliers) between features and targets. Because many pairs are tested at once, p-values are adjusted with the **Benjamini–Hochberg false discovery rate**; only `q < 0.05` should be called significant.

In [ ]:
corr_vars = [c for c in DEPRIVATION + SOCIO + ['dep_index_z', 'margin_2021', 'enp_2021', 'share_anc_2021',
                                                'retained', 'turnout_2016'] if c in master]
rows = []
for tgt in TARGETS:
    for v in corr_vars:
        d = master[[tgt, v]].dropna()
        if len(d) > 10 and d[v].nunique() > 1:
            rho, p = stats.spearmanr(d[tgt], d[v])
            rows.append({'target': tgt, 'feature': v, 'rho': rho, 'p': p, 'n': len(d)})
corr_tab = pd.DataFrame(rows)
corr_tab['q_fdr'] = multipletests(corr_tab['p'], method='fdr_bh')[1]
corr_tab['significant'] = corr_tab['q_fdr'] < 0.05
display(corr_tab.sort_values(['target', 'rho']).round(4))

plt.figure(figsize=(11, 8))
cm = master[list(dict.fromkeys(TARGETS + corr_vars))].corr(method='spearman')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot_kws={'size': 8})
plt.title('Spearman correlation matrix'); plt.tight_layout(); plt.show()

### 7.5 Multicollinearity (VIF)
Deprivation indicators move together. A **variance inflation factor** above ~5 means a coefficient's standard error is badly inflated, so individual betas can't be trusted to separate effects. This is the statistical reason to (a) combine indicators into an index for inference and (b) prefer **Ridge** (which handles correlated predictors) over plain OLS for prediction.

In [ ]:
vif_vars = [c for c in DEPRIVATION + SOCIO + ['margin_2021', 'enp_2021'] if c in master]
Xv = master[vif_vars].dropna()
Xv = sm.add_constant((Xv - Xv.mean()) / Xv.std())
vif = pd.Series([variance_inflation_factor(Xv.values, i) for i in range(1, Xv.shape[1])],
                index=vif_vars, name='VIF').sort_values(ascending=False)
display(vif.round(2).to_frame())
RESULTS['max_vif'] = float(vif.max())

### 7.6 Is the deprivation index internally consistent?
**Cronbach's α** checks whether the components measure one underlying construct (α ≥ 0.7 is conventionally acceptable). Together with the PCA variance explained (Section 6), this tells us whether a single index is a defensible summary or whether components should be analysed separately.

In [ ]:
def cronbach_alpha(df):
    df = df.dropna(); k = df.shape[1]
    return k / (k - 1) * (1 - df.var(ddof=1).sum() / df.sum(axis=1).var(ddof=1))
alpha = cronbach_alpha(Z)
print(f"Cronbach's alpha ({len(INDEX_COMPONENTS)} components): {alpha:.3f}")
print(f"PCA PC1 variance explained: {RESULTS['pca_var_explained']:.1%}")
print(f"Correlation z-index vs PCA index: {master[['dep_index_z','dep_index_pca']].corr().iloc[0,1]:.3f}")
RESULTS['cronbach_alpha'] = float(alpha)

---
## 8. Exploratory Data Analysis
### 8.1 Distributions of the targets

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS) + 1, figsize=(5 * (len(TARGETS) + 1), 4))
for ax, tgt in zip(axes, TARGETS + ['d_turnout']):
    sns.histplot(data=master, x=tgt, hue='metro', kde=True, ax=ax, element='step')
    ax.set_title(tgt)
plt.tight_layout(); plt.show()

Look for: the gap between metros, long tails (very low-turnout wards are the policy-relevant ones), and whether `d_turnout` is mostly negative (a metro-wide decline between 2016 and 2021 means *relative* differences across wards matter more than the average drop).

### 8.2 Turnout against each deprivation dimension
Each panel shows wards as points with a LOWESS smoother, which reveals non-linearity (e.g. turnout only falling after informal housing passes some threshold). Non-linear shapes are an argument for tree models; straight lines favour linear models.

In [ ]:
plot_vars = [c for c in DEPRIVATION + SOCIO + ['dep_index_z'] if c in master]
ncol = 3; nrow = int(np.ceil(len(plot_vars) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 4 * nrow)); axes = np.ravel(axes)
for ax, v in zip(axes, plot_vars):
    for m, d in master.groupby('metro'):
        ax.scatter(d[v], d['turnout_2021'], s=14, alpha=.55, label=m)
    d = master[[v, 'turnout_2021']].dropna()
    lw = sm.nonparametric.lowess(d['turnout_2021'], d[v], frac=.6)
    ax.plot(lw[:, 0], lw[:, 1], color='black', lw=2)
    ax.set_xlabel(v); ax.set_ylabel('turnout 2021')
for ax in axes[len(plot_vars):]: ax.axis('off')
axes[0].legend(); plt.tight_layout(); plt.show()

### 8.3 Political landscape: winners, dominance and turnout

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.boxplot(data=master, x='winner_grp_2021', y='turnout_2021', hue='metro', ax=axes[0])
axes[0].set_title('Turnout by 2021 winning party')
sns.scatterplot(data=master, x='enp_2021', y='turnout_2021', hue='winner_grp_2021', ax=axes[1])
axes[1].set_title('Party fragmentation (ENP) vs turnout')
sns.scatterplot(data=master, x='margin_2021', y='turnout_2021', hue='metro', ax=axes[2])
axes[2].set_title('Winning margin vs turnout')
plt.tight_layout(); plt.show()
display(pd.crosstab(master['winner_2016'], master['winner_2021'], margins=True))

The crosstab is the raw material for H2: the diagonal = wards retained by the same party; off-diagonal = wards that flipped. If almost every ward is on the diagonal, the retained/flipped comparison has very few "flipped" cases and its tests will have low power — note this honestly in the presentation.

### 8.4 Disengagement vs defection (visual test of H2)
Each point is a ward. **x** = change in the 2016 winner's vote share (defection when negative); **y** = change in turnout (disengagement when negative). H2 predicts retained wards cluster where turnout falls a lot while the incumbent's share barely moves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=master, x='d_incumbent_share', y='d_turnout', hue='retained', style='metro', ax=axes[0])
axes[0].axhline(0, c='grey', ls='--'); axes[0].axvline(0, c='grey', ls='--')
axes[0].set_title('Change in incumbent share vs change in turnout')
sns.boxplot(data=master.dropna(subset=['retained']), x='retained', y='d_turnout', ax=axes[1])
axes[1].set_title('Turnout change: flipped (0) vs retained (1)')
plt.tight_layout(); plt.show()

### 8.5 Where are the lowest-participation wards?

In [ ]:
cols = ['ward_id', 'metro', 'turnout_2021', 'd_turnout', 'winner_2021', 'dep_index_z'] + \
       [c for c in ['youth_reg_share_2026'] if c in master]
print('Lowest 10 turnout wards'); display(master.nsmallest(10, 'turnout_2021')[cols])
print('Highest 10 turnout wards'); display(master.nlargest(10, 'turnout_2021')[cols])

---
## 9. Hypothesis Testing
All regressions use **standardised variables** (so coefficients are effect sizes in standard deviations and directly comparable — exactly what H1 asks) and **HC3 robust standard errors** (valid even if error variance differs across wards).

In [ ]:
def ols_std(df, y, xs, cats=()):
    d = df[[y] + list(xs) + list(cats)].dropna().copy()
    for c in [y] + list(xs):
        d[c] = (d[c] - d[c].mean()) / d[c].std()
    f = f'{y} ~ ' + ' + '.join(list(xs) + [f'C({c})' for c in cats])
    return smf.ols(f, d).fit(cov_type='HC3'), d

def coef_table(m):
    ci = m.conf_int()
    return pd.DataFrame({'beta': m.params, 'se': m.bse, 'p': m.pvalues, 'ci_low': ci[0], 'ci_high': ci[1]}).round(4)

### 9.1 H1 — Deprivation friction
**Test design.**
1. Model A: target ~ deprivation index + metro. Model B: target ~ each component + metro.
2. If income is available: target ~ deprivation index + income + metro, then **bootstrap** (2,000 resamples) the difference |β_deprivation| − |β_income|. H1 is supported if that difference's 95% CI is above zero.
3. Run for every available target (T1 overall turnout, T2 youth registration).

In [ ]:
H1 = {}
for tgt in TARGETS:
    print(f'\n######## Target: {tgt} ########')
    mA, dA = ols_std(master, tgt, ['dep_index_z'], ['metro'])
    print(f'Model A  beta(dep_index)={mA.params["dep_index_z"]:.3f}  p={mA.pvalues["dep_index_z"]:.4f}  R2={mA.rsquared:.3f}')
    comps = [c for c in INDEX_COMPONENTS if master[c].notna().any()]
    mB, _ = ols_std(master, tgt, comps + [c for c in ['youth_share_adults'] if c in master], ['metro'])
    print('Model B (components):'); display(coef_table(mB))
    bp = het_breuschpagan(mB.resid, mB.model.exog)
    print(f'Breusch-Pagan heteroscedasticity p={bp[1]:.4f} (HC3 SEs already guard against this)')
    H1[tgt] = {'beta_dep': float(mA.params['dep_index_z']), 'p_dep': float(mA.pvalues['dep_index_z']),
               'r2_A': float(mA.rsquared)}
    if 'median_income' in SOCIO:
        mC, dC = ols_std(master, tgt, ['dep_index_z', 'median_income'], ['metro'])
        display(coef_table(mC))
        rng = np.random.default_rng(RNG); diffs = []
        for _ in range(2000):
            s = dC.sample(len(dC), replace=True, random_state=int(rng.integers(1e9)))
            m = smf.ols(mC.model.formula, s).fit()
            diffs.append(abs(m.params['dep_index_z']) - abs(m.params['median_income']))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        print(f'|beta_dep| - |beta_income| 95% bootstrap CI: [{lo:.3f}, {hi:.3f}]')
        H1[tgt].update({'diff_ci': (float(lo), float(hi))})
    else:
        print('median_income not available -> the income comparison in H1 cannot be run. '
              'Verify whether a ward-level income table exists for Census 2022; otherwise use a proxy '
              '(e.g. Census 2011 income, GCRO Quality of Life survey) and state the limitation.')
RESULTS['H1'] = H1

### 9.2 H2 — Incumbency demobilisation
**Test design.**
1. **Group test:** Mann–Whitney on `d_turnout` for retained vs flipped wards. H2 predicts a *larger decline* in retained wards.
2. **Controlled test:** turnout_2021 ~ retained + turnout_2016 + deprivation + metro. Controlling for 2016 turnout makes the retained coefficient a statement about *change*.
3. **Disengagement vs defection:** in retained wards, compare the average turnout change with the average incumbent-share change. H2's mechanism is supported if turnout falls clearly while the incumbent's share is stable (people stay home rather than switch).

Caveat: with only two elections, "prolonged tenure" is measured as holding the ward in both 2016 and 2021. Adding 2011 (and 2006) results through the same voting-district crosswalk would give true tenure length.

In [ ]:
H2 = {}
d2 = master.dropna(subset=['retained', 'd_turnout'])
ret, flip = d2.loc[d2.retained == 1, 'd_turnout'], d2.loc[d2.retained == 0, 'd_turnout']
print(f'Retained wards: {len(ret)} | Flipped wards: {len(flip)}')
if len(flip) >= 5:
    u = stats.mannwhitneyu(ret, flip, alternative='less')
    eff = 1 - 2 * u.statistic / (len(ret) * len(flip))       # rank-biserial correlation
    print(f'Median d_turnout retained={ret.median():.3f}, flipped={flip.median():.3f}; '
          f'one-sided Mann-Whitney p={u.pvalue:.4f}, rank-biserial r={eff:.3f}')
    H2['mw_p'] = float(u.pvalue); H2['rank_biserial'] = float(eff)
else:
    print('Too few flipped wards for a reliable group test.')

m2, _ = ols_std(master, 'turnout_2021', ['retained', 'turnout_2016', 'dep_index_z'], ['metro'])
display(coef_table(m2))
H2['beta_retained'] = float(m2.params['retained']); H2['p_retained'] = float(m2.pvalues['retained'])

m2b, _ = ols_std(master, 'turnout_2021', ['anc_retained', 'turnout_2016', 'dep_index_z'], ['metro'])
print(f'ANC-retained specifically: beta={m2b.params["anc_retained"]:.3f}, p={m2b.pvalues["anc_retained"]:.4f}')

r_only = d2[d2.retained == 1]
dt, ds = r_only['d_turnout'], r_only['d_incumbent_share']
print(f'In retained wards: mean d_turnout={dt.mean():+.3f} (t-test vs 0 p={stats.ttest_1samp(dt, 0).pvalue:.4f}); '
      f'mean d_incumbent_share={ds.mean():+.3f} (p={stats.ttest_1samp(ds.dropna(), 0).pvalue:.4f})')
print(f'Share of retained wards where |turnout drop| > |incumbent share drop|: '
      f'{(dt.abs() > ds.abs()).mean():.1%}')
H2.update({'mean_dturnout_retained': float(dt.mean()), 'mean_dshare_retained': float(ds.mean())})
RESULTS['H2'] = H2

---
## 10. Predictive Modelling — Regression (target: ward turnout)
### 10.1 A forecasting design that can actually be used for 2026
To forecast 2026 honestly, the model may only use information available **before** an election:

| | Training | Forecasting |
|---|---|---|
| Target | turnout 2021 | turnout 2026 |
| Structural features | Census 2022 deprivation & age | Census 2022 deprivation & age |
| Lagged political features | 2016 results (reconciled to 2021 wards) | 2021 results |

Lagged features are renamed `lag_*` so the same fitted model applies in both columns. Contemporaneous 2021 results (winner, margin in 2021) are **excluded** from the predictive model because they are not known before election day — using them would be target leakage.

In [ ]:
LAG_VARS = ['turnout', 'margin', 'enp', 'share_anc', 'share_da', 'spoilt_rate']
# Components only: the index is a linear combination of them, so including both adds pure collinearity
STRUCT_FEATS = [c for c in DEPRIVATION + SOCIO if c in master and master[c].notna().any()]

def make_feature_frame(df, lag_year):
    X = df[STRUCT_FEATS].copy()
    for v in LAG_VARS:
        X[f'lag_{v}'] = df[f'{v}_{lag_year}'] if f'{v}_{lag_year}' in df else np.nan
    X['lag_winner'] = df[f'winner_grp_{lag_year}']
    X['metro'] = df['metro']
    return X

NUM_FEATS = STRUCT_FEATS + [f'lag_{v}' for v in LAG_VARS]
CAT_FEATS = ['lag_winner', 'metro']
TARGET = 'turnout_2021'
model_df = master[master[TARGET].notna()].reset_index(drop=True)
X = make_feature_frame(model_df, 2016)
y = model_df[TARGET]
print('X shape:', X.shape, '| features:', NUM_FEATS + CAT_FEATS)

### 10.2 Spatial cross-validation
Neighbouring wards share conditions, so a random split lets the model "peek" at a test ward through its neighbours in training, inflating scores. We therefore split by **spatial blocks**: with a centroids file, K-means clusters of ward centroids; otherwise blocks of 8 consecutive ward numbers within a metro (ward numbering is broadly contiguous in both metros — a centroid file is better and is picked up automatically if present). The whole 5-fold procedure is **repeated 3 times** with different block-to-fold assignments (15 test folds) for stable estimates.

In [ ]:
cent_path = os.path.join(BASE_DIR, CENTROIDS_FILE)
if os.path.exists(cent_path):
    from sklearn.cluster import KMeans
    cent = standardize_ward_id(read_csv_safe(cent_path), None, CENTROIDS_FILE).set_index('ward_id')
    xy = cent.reindex(model_df.ward_id)[['lat', 'lon']]
    groups = pd.Series(KMeans(30, random_state=RNG, n_init=10).fit_predict(xy.fillna(xy.mean())), index=model_df.index)
    print('Spatial groups: K-means on centroids')
else:
    groups = model_df['metro'].str[:3] + '_' + (model_df['ward_num'] // 8).astype(str)
    print('Spatial groups: ward-number blocks of 8 (add a centroids file for true geography)')
print('Number of spatial blocks:', groups.nunique())

def repeated_group_kfold(groups, n_splits=5, n_repeats=3, seed=RNG):
    uniq = np.array(sorted(pd.unique(groups)))
    rng = np.random.default_rng(seed)
    for r in range(n_repeats):
        perm = rng.permutation(uniq)
        for k in range(n_splits):
            test_g = set(perm[k::n_splits])
            te = np.where(groups.isin(test_g))[0]; tr = np.where(~groups.isin(test_g))[0]
            yield r, k, tr, te
SPLITS = list(repeated_group_kfold(groups))

### 10.3 Candidate models — and why each is on the list
| Model | Why include it |
|---|---|
| Baseline (mean) | Any real model must beat "predict the average". |
| OLS | Transparent; coefficients = effect sizes; the inferential benchmark. |
| Ridge | OLS plus shrinkage; designed for correlated predictors (see VIF, 7.5) and small n. |
| Lasso | Shrinkage that can zero out weak features — a check on which features matter. |
| Random Forest | Captures non-linearities and interactions; robust to outliers. |
| Gradient Boosting (sklearn) | Strong tabular learner; shallow trees + small learning rate for small n. |
| XGBoost | The model named in the proposal; regularised boosting. |

All models share one preprocessing pipeline: median imputation with **missing-value indicators** (fitted on training folds only — fixing leakage issue #6), scaling, and one-hot encoding of the lagged winner and metro. Tree hyper-parameters are deliberately conservative (shallow depth, large leaves) because with ~240 wards deep trees memorise noise.

In [ ]:
def make_prep():
    num = Pipeline([('imp', SimpleImputer(strategy='median', add_indicator=True)), ('sc', StandardScaler())])
    cat = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                    ('oh', OneHotEncoder(handle_unknown='ignore', drop='if_binary'))])
    return ColumnTransformer([('num', num, NUM_FEATS), ('cat', cat, CAT_FEATS)])

def P(model): return Pipeline([('prep', make_prep()), ('model', model)])

REG_MODELS = {
    'Baseline (mean)':   P(DummyRegressor()),
    'OLS':               P(LinearRegression()),
    'Ridge':             P(RidgeCV(alphas=np.logspace(-3, 3, 30))),
    'Lasso':             P(LassoCV(cv=5, random_state=RNG, max_iter=50000)),
    'Random Forest':     P(RandomForestRegressor(n_estimators=500, min_samples_leaf=5, max_features=0.5,
                                                 random_state=RNG, n_jobs=-1)),
    'Gradient Boosting': P(GradientBoostingRegressor(n_estimators=300, learning_rate=0.03, max_depth=2,
                                                     subsample=0.8, random_state=RNG)),
}
if HAS_XGB:
    REG_MODELS['XGBoost'] = P(XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=3, subsample=0.8,
                                           colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0,
                                           random_state=RNG, n_jobs=-1, verbosity=0))
COMPLEXITY_ORDER = list(REG_MODELS)   # simplest first: used by the one-SE rule

### 10.4 Run the cross-validation
Metrics: **MAE** (average error in turnout percentage points — easiest to explain to a non-technical panel), **RMSE** (penalises big misses), **R²** (share of variance explained). Out-of-fold predictions from the first repeat are kept for diagnostics.

In [ ]:
def cv_regression(models, X, y, splits):
    rows, oof = [], {}
    for name, est in models.items():
        pred = np.full(len(y), np.nan)
        for r, k, tr, te in splits:
            m = clone(est).fit(X.iloc[tr], y.iloc[tr])
            p = m.predict(X.iloc[te])
            if r == 0: pred[te] = p
            rows.append({'model': name, 'repeat': r, 'fold': k,
                         'MAE': mean_absolute_error(y.iloc[te], p),
                         'RMSE': mean_squared_error(y.iloc[te], p) ** 0.5,
                         'R2': r2_score(y.iloc[te], p)})
        oof[name] = pred
    return pd.DataFrame(rows), oof

reg_folds, reg_oof = cv_regression(REG_MODELS, X, y, SPLITS)
reg_summary = reg_folds.groupby('model').agg(MAE_mean=('MAE', 'mean'), MAE_sd=('MAE', 'std'),
                                             RMSE_mean=('RMSE', 'mean'), R2_mean=('R2', 'mean'), R2_sd=('R2', 'std'))
reg_summary['MAE_se'] = reg_summary['MAE_sd'] / np.sqrt(len(SPLITS))
reg_summary = reg_summary.loc[COMPLEXITY_ORDER]
reg_summary['MAE_pp'] = reg_summary['MAE_mean'] * 100
display(reg_summary.round(4))

plt.figure(figsize=(10, 4.5))
sns.boxplot(data=reg_folds, x='model', y='MAE', order=COMPLEXITY_ORDER)
plt.xticks(rotation=25); plt.title('Spatial-CV MAE per fold (lower is better)'); plt.tight_layout(); plt.show()

### 10.5 How much does ignoring geography inflate scores?
Same models, ordinary random 5-fold CV. If random CV looks clearly better than spatial CV, the gap is the overfitting the proposal warned about — evidence that spatial CV is the right yardstick.

In [ ]:
rand_splits = [(0, k, tr, te) for k, (tr, te) in enumerate(KFold(5, shuffle=True, random_state=RNG).split(X))]
cmp_models = {k: REG_MODELS[k] for k in ['Ridge', 'Random Forest'] + (['XGBoost'] if HAS_XGB else [])}
rand_folds, _ = cv_regression(cmp_models, X, y, rand_splits)
gap = pd.DataFrame({'R2_random_CV': rand_folds.groupby('model')['R2'].mean(),
                    'R2_spatial_CV': reg_summary.loc[list(cmp_models), 'R2_mean']})
gap['optimism'] = gap['R2_random_CV'] - gap['R2_spatial_CV']
display(gap.round(4))

---
## 11. Predictive Modelling — Classification (flagging low-turnout wards)
**Target:** `low_turnout` = 1 if a ward is in the bottom third of 2021 turnout. **Use case:** a ranked list of "demobilisation-risk" wards for registration and GOTV drives.

Metrics: **ROC-AUC** (how well wards are ranked by risk — the main metric), **balanced accuracy** and **F1** (at a 0.5 threshold, with class weighting), **Brier score** (calibration of probabilities). Same features, same spatial folds.

The final row answers "classification or regression?" with data: it scores the **regression model's out-of-fold predictions** as a risk ranking. If it matches the dedicated classifiers' AUC, one regression model serves both purposes.

In [ ]:
cut = y.quantile(1 / 3)
y_cls = (y <= cut).astype(int)
print(f'Low-turnout threshold: turnout <= {cut:.3f}  | positives: {y_cls.sum()} / {len(y_cls)}')

CLS_MODELS = {
    'Baseline (prior)':    P(DummyClassifier(strategy='prior')),
    'Logistic (L2)':       P(LogisticRegression(C=1.0, max_iter=5000, class_weight='balanced')),
    'Random Forest':       P(RandomForestClassifier(n_estimators=500, min_samples_leaf=5, max_features=0.5,
                                                    class_weight='balanced', random_state=RNG, n_jobs=-1)),
    'Gradient Boosting':   P(GradientBoostingClassifier(n_estimators=300, learning_rate=0.03, max_depth=2,
                                                        subsample=0.8, random_state=RNG)),
}
if HAS_XGB:
    CLS_MODELS['XGBoost'] = P(XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=3, subsample=0.8,
                                            colsample_bytree=0.8, min_child_weight=5, random_state=RNG,
                                            n_jobs=-1, verbosity=0, eval_metric='logloss'))
rows = []
for name, est in CLS_MODELS.items():
    for r, k, tr, te in SPLITS:
        if y_cls.iloc[te].nunique() < 2: continue
        m = clone(est).fit(X.iloc[tr], y_cls.iloc[tr])
        pr = m.predict_proba(X.iloc[te])[:, 1]
        rows.append({'model': name, 'AUC': roc_auc_score(y_cls.iloc[te], pr),
                     'BalAcc': balanced_accuracy_score(y_cls.iloc[te], (pr >= .5).astype(int)),
                     'F1': f1_score(y_cls.iloc[te], (pr >= .5).astype(int), zero_division=0),
                     'Brier': brier_score_loss(y_cls.iloc[te], pr)})
cls_folds = pd.DataFrame(rows)
cls_summary = cls_folds.groupby('model').agg(['mean', 'std']).round(4)

# Regression-as-ranker: lower predicted turnout = higher risk (first-repeat OOF predictions)
best_reg_name = reg_summary.drop('Baseline (mean)')['MAE_mean'].idxmin()
auc_from_reg = [roc_auc_score(y_cls.iloc[te], -reg_oof[best_reg_name][te])
                for r, k, tr, te in SPLITS if r == 0 and y_cls.iloc[te].nunique() == 2]
display(cls_summary)
print(f'Regression ({best_reg_name}) used as risk ranker: AUC = {np.mean(auc_from_reg):.4f} ± {np.std(auc_from_reg):.4f}')
RESULTS['cls_best_auc'] = float(cls_summary[('AUC', 'mean')].drop('Baseline (prior)').max())
RESULTS['reg_as_ranker_auc'] = float(np.mean(auc_from_reg))

---
## 12. Model Selection — Which Model and Why
### 12.1 Decision rule, fixed before looking at results
With ~240 wards, differences of a fraction of a percentage point between models are within noise. We use the **one-standard-error rule** (Breiman et al.; Hastie, Tibshirani & Friedman): among all models whose mean spatial-CV MAE is within one standard error of the best, choose the **simplest**. Rationale: equal accuracy, more interpretable, more stable, less likely to overfit on new (2026) data. The rule also requires the chosen model to beat the baseline.

**Stability rule:** if any model feature has a VIF above 10, plain OLS is excluded, because its coefficients would swing wildly between samples even when its predictions are fine; Ridge then becomes the simplest eligible model.

In [ ]:
s = reg_summary.drop('Baseline (mean)')
best = s['MAE_mean'].idxmin()
threshold = s.loc[best, 'MAE_mean'] + s.loc[best, 'MAE_se']
eligible = [m for m in COMPLEXITY_ORDER if m in s.index and s.loc[m, 'MAE_mean'] <= threshold]

# Stability rule: plain OLS is only acceptable if its predictors are not badly collinear
Xn = X[NUM_FEATS].dropna()
Xn = sm.add_constant((Xn - Xn.mean()) / Xn.std())
model_vif = pd.Series([variance_inflation_factor(Xn.values, i) for i in range(1, Xn.shape[1])], index=NUM_FEATS)
print('VIF of model features:', model_vif.round(1).to_dict())
if model_vif.max() > 10 and 'OLS' in eligible and len(eligible) > 1:
    eligible.remove('OLS')
    print(f'Max VIF {model_vif.max():.1f} > 10 -> OLS coefficients unstable; OLS excluded, shrinkage models preferred.')
CHOSEN = eligible[0]
base_mae = reg_summary.loc['Baseline (mean)', 'MAE_mean']
print(f'Lowest-MAE model: {best} (MAE {s.loc[best,"MAE_mean"]*100:.2f} pp ± SE {s.loc[best,"MAE_se"]*100:.2f})')
print(f'One-SE threshold: {threshold*100:.2f} pp -> eligible: {eligible}')
print(f'CHOSEN MODEL: {CHOSEN} | MAE {s.loc[CHOSEN,"MAE_mean"]*100:.2f} pp, R2 {s.loc[CHOSEN,"R2_mean"]:.3f}')
print(f'Improvement over baseline: {(1 - s.loc[CHOSEN,"MAE_mean"]/base_mae):.1%} lower MAE')

# Paired fold-wise comparison of chosen vs best
if CHOSEN != best:
    a = reg_folds[reg_folds.model == CHOSEN].sort_values(['repeat', 'fold'])['MAE'].values
    b = reg_folds[reg_folds.model == best].sort_values(['repeat', 'fold'])['MAE'].values
    print(f'Paired Wilcoxon {CHOSEN} vs {best}: p={stats.wilcoxon(a, b).pvalue:.4f} '
          f'(p>0.05 = no evidence the complex model is better)')
RESULTS.update({'chosen_model': CHOSEN, 'best_mae_model': best,
                'chosen_mae_pp': float(s.loc[CHOSEN, 'MAE_mean'] * 100), 'chosen_r2': float(s.loc[CHOSEN, 'R2_mean']),
                'baseline_mae_pp': float(base_mae * 100)})

### 12.2 How to justify the choice in the presentation
The printed output above is the evidence. Frame it like this, filling in the real numbers:

1. **Accuracy:** "Model X predicts ward turnout within *N* percentage points on wards it never saw, in held-out geographic blocks — *M*% better than predicting the average."
2. **Parsimony:** "More complex models (XGBoost, forests) were not reliably better — within one standard error — so we chose the simpler model."
3. **Fit for purpose:** the challenge asks *to what extent* factors drive participation; a linear/regularised model gives signed, standardised effects that answer that directly, while tree models need post-hoc explanations.
4. **Robustness:** spatial CV (10.5) shows how much random CV would have flattered us.

**If XGBoost *does* win clearly** (outside one SE, and the Wilcoxon test is significant), the rule picks it — that is evidence of real non-linearity (check the LOWESS curves in 8.2), and Section 13's permutation importance and partial-dependence plots provide the explanation layer.

**Regression vs classification:** compare the classifier AUCs with the "regression as ranker" AUC in Section 11. If they are similar, keep **one regression model** for both the forecast and the risk list — simpler to deploy and it keeps the full turnout information.

### 12.3 Diagnostics for the chosen model
Out-of-fold predicted vs actual (points should hug the diagonal), residuals vs predicted (no funnel or curve), and residuals by metro (no systematic bias for one city).

In [ ]:
pred = reg_oof[CHOSEN]; resid = y - pred
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].scatter(y, pred, s=14, alpha=.6); lim = [min(y.min(), pred.min()), max(y.max(), pred.max())]
axes[0].plot(lim, lim, 'k--'); axes[0].set_xlabel('actual'); axes[0].set_ylabel('predicted (out-of-fold)')
axes[0].set_title(f'{CHOSEN}: predicted vs actual')
axes[1].scatter(pred, resid, s=14, alpha=.6); axes[1].axhline(0, c='k', ls='--')
axes[1].set_xlabel('predicted'); axes[1].set_ylabel('residual'); axes[1].set_title('Residuals vs predicted')
sns.boxplot(x=model_df['metro'], y=resid, ax=axes[2]); axes[2].axhline(0, c='k', ls='--')
axes[2].set_title('Residuals by metro')
plt.tight_layout(); plt.show()
print('Mean residual by metro:', resid.groupby(model_df['metro']).mean().round(4).to_dict())

---
## 13. Interpretation
### 13.1 Fit the chosen model on all wards; coefficients (linear models)
Features are standardised inside the pipeline, so linear coefficients are "change in turnout (as a proportion) per one standard deviation of the feature", holding the others constant.

In [ ]:
final_model = clone(REG_MODELS[CHOSEN]).fit(X, y)
feat_names = final_model.named_steps['prep'].get_feature_names_out()
est = final_model.named_steps['model']
if hasattr(est, 'coef_'):
    coefs = pd.Series(np.ravel(est.coef_), index=feat_names).sort_values()
    plt.figure(figsize=(8, 0.35 * len(coefs) + 1)); coefs.plot.barh(color=np.where(coefs < 0, '#c0392b', '#2471a3'))
    plt.title(f'{CHOSEN} coefficients (per 1 SD)'); plt.tight_layout(); plt.show()
    display(coefs.to_frame('coef').round(4))
else:
    print(f'{CHOSEN} has no coefficients — see permutation importance below.')

### 13.2 Permutation importance (model-agnostic, on held-out folds)
For each test fold, shuffle one feature and measure how much MAE worsens. Computed on **held-out** wards, so it reflects predictive value, not memorisation. Works identically for linear and tree models, which lets us compare them fairly.

In [ ]:
imp_rows = []
for r, k, tr, te in [s_ for s_ in SPLITS if s_[0] == 0]:
    m = clone(REG_MODELS[CHOSEN]).fit(X.iloc[tr], y.iloc[tr])
    pi = permutation_importance(m, X.iloc[te], y.iloc[te], n_repeats=15, random_state=RNG,
                                scoring='neg_mean_absolute_error')
    imp_rows.append(pd.Series(pi.importances_mean, index=X.columns))
imp = pd.concat(imp_rows, axis=1)
imp_summary = pd.DataFrame({'mean_increase_in_MAE_pp': imp.mean(axis=1) * 100,
                            'sd': imp.std(axis=1) * 100}).sort_values('mean_increase_in_MAE_pp')
plt.figure(figsize=(8, 0.35 * len(imp_summary) + 1))
plt.barh(imp_summary.index, imp_summary.mean_increase_in_MAE_pp, xerr=imp_summary.sd, color='#5d6d7e')
plt.xlabel('MAE increase when shuffled (percentage points)'); plt.title('Permutation importance (held-out)')
plt.tight_layout(); plt.show()
RESULTS['top_features'] = imp_summary.sort_values('mean_increase_in_MAE_pp', ascending=False).head(5).index.tolist()
print('Top features:', RESULTS['top_features'])

### 13.3 Partial dependence — the shape of each relationship
Average predicted turnout as one feature varies, others held at observed values. Flat = no effect; slope = direction; kinks = thresholds (policy-relevant: e.g. turnout drops once informal housing exceeds some share).

In [ ]:
top_num = [f for f in RESULTS['top_features'] if f in NUM_FEATS][:4]
if top_num:
    fig, ax = plt.subplots(figsize=(4 * len(top_num), 4))
    PartialDependenceDisplay.from_estimator(final_model, X, top_num, ax=ax)
    plt.tight_layout(); plt.show()

---
## 14. 2026 Forecast & Policy Simulator
### 14.1 Ward-level turnout forecast for 4 November 2026
Same model, with lagged features now taken from **2021**. The 90% interval comes from the distribution of out-of-fold residuals (a split-conformal approximation), so it reflects how wrong the model actually was on unseen wards.

**Caveats:** (1) The model learned the 2016→2021 relationship; national mood in 2026 can shift every ward together — the *ranking* of wards is more reliable than the level. (2) Check the Municipal Demarcation Board's final ward delimitation for the 2026 elections; if boundaries changed, map forecasts to the new wards through voting districts as in 5.5. (3) By-election results (2021–2026) can be added as a feature once collected for enough wards.

In [ ]:
X26 = make_feature_frame(master, 2021)
forecast = master[['ward_id', 'metro', 'turnout_2021']].copy()
forecast['pred_turnout_2026'] = final_model.predict(X26)
q_lo, q_hi = np.nanpercentile(resid, [5, 95])
forecast['pi90_low'] = (forecast.pred_turnout_2026 + q_lo).clip(0, 1)
forecast['pi90_high'] = (forecast.pred_turnout_2026 + q_hi).clip(0, 1)
forecast['pred_change'] = forecast.pred_turnout_2026 - forecast.turnout_2021
for c in ['youth_reg_share_2026', 'youth_reg_rate_2026', 'dep_index_z']:
    if c in master: forecast[c] = master[c]
display(forecast.groupby('metro')[['turnout_2021', 'pred_turnout_2026']].mean().round(4))
display(forecast.nsmallest(10, 'pred_turnout_2026').round(3))

### 14.2 Priority wards: low forecast turnout *and* weak youth registration
Combines the forecast with the youth-specific registration measure (when available) — the intersection is where youth mobilisation effort has the most room.

In [ ]:
if 'youth_reg_share_2026' in forecast:
    lo_t = forecast.pred_turnout_2026 <= forecast.pred_turnout_2026.quantile(1/3)
    lo_y = forecast.youth_reg_share_2026 <= forecast.youth_reg_share_2026.quantile(1/3)
    forecast['priority'] = np.select([lo_t & lo_y, lo_t | lo_y], ['HIGH', 'MEDIUM'], 'LOW')
    display(forecast.priority.value_counts().to_frame())
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=forecast, x='youth_reg_share_2026', y='pred_turnout_2026', hue='priority', style='metro',
                    palette={'HIGH': '#c0392b', 'MEDIUM': '#e67e22', 'LOW': '#95a5a6'})
    plt.title('Forecast turnout vs youth registration share'); plt.tight_layout(); plt.show()
else:
    lo_t = forecast.pred_turnout_2026 <= forecast.pred_turnout_2026.quantile(1/3)
    forecast['priority'] = np.where(lo_t, 'HIGH', 'LOW')
    print('Registration data missing: priority based on forecast turnout only.')

### 14.3 Policy simulator (what the Streamlit app will expose)
Changes deprivation inputs and re-predicts. Percentage features are clipped to [0, 1]; the deprivation index is recomputed so it stays consistent with its components. **These are model-implied associations, not causal effects** — present them as "wards like this, with better services, tend to have turnout about X points higher".

In [ ]:
comp_stats = {c: (master[c].mean(), master[c].std()) for c in INDEX_COMPONENTS}

def simulate_policy(model, Xbase, changes):
    X2 = Xbase.copy()
    for col, delta in changes.items():
        if col in X2:
            X2[col] = (X2[col] + delta).clip(0, 1) if col.startswith('pct_') else X2[col] + delta
    if 'dep_index_z' in X2:
        X2['dep_index_z'] = np.nanmean(np.column_stack(
            [(X2[c] - m) / s for c, (m, s) in comp_stats.items() if c in X2]), axis=1)
    return model.predict(X2) - model.predict(Xbase)

scenarios = {
    'Informal dwellings -10pp':      {'pct_informal_dwelling': -0.10},
    'Weekly refuse for +10pp hh':    {'pct_refuse_not_weekly': -0.10},
    'Piped water in dwelling +10pp': {'pct_water_outside_dwelling': -0.10},
    'All three combined':            {'pct_informal_dwelling': -0.10, 'pct_refuse_not_weekly': -0.10,
                                      'pct_water_outside_dwelling': -0.10},
}
sim = pd.DataFrame({name: simulate_policy(final_model, X26, ch) for name, ch in scenarios.items()})
sim['metro'] = master['metro'].values
display((sim.groupby('metro').mean() * 100).round(2).T.rename_axis('Mean turnout change (pp)'))

---
## 15. Export & Conclusions
### 15.1 Save artefacts for the Streamlit app and the submission

In [ ]:
OUT_DIR = '/tmp/civicpulse_outputs/' if USE_SYNTHETIC else os.path.join(os.path.dirname(BASE_DIR.rstrip('/')), 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
master.to_csv(os.path.join(OUT_DIR, 'master_ward_table.csv'), index=False)
forecast.to_csv(os.path.join(OUT_DIR, 'forecast_2026.csv'), index=False)
reg_summary.to_csv(os.path.join(OUT_DIR, 'regression_cv_results.csv'))
joblib.dump(final_model, os.path.join(OUT_DIR, 'turnout_model.joblib'))
meta = {'chosen_model': CHOSEN, 'target': TARGET, 'num_features': NUM_FEATS, 'cat_features': CAT_FEATS,
        'index_components': INDEX_COMPONENTS, 'component_stats': comp_stats,
        'residual_q05_q95': [float(q_lo), float(q_hi)], 'synthetic': USE_SYNTHETIC}
json.dump(meta, open(os.path.join(OUT_DIR, 'model_metadata.json'), 'w'), indent=2, default=float)
print('Saved to', OUT_DIR, '->', sorted(os.listdir(OUT_DIR)))

### 15.2 Data-driven verdicts
This cell reads the stored results and applies the decision criteria stated in Sections 9 and 12, so the conclusions come from the numbers rather than from the narrative.

In [ ]:
def verdict(cond, yes, no): return yes if cond else no
print('=' * 78)
if USE_SYNTHETIC: print('!!! SYNTHETIC DATA — verdicts below only demonstrate the logic. !!!')
print('=' * 78)
for tgt, h in RESULTS['H1'].items():
    line = f"H1 [{tgt}]: beta(deprivation) = {h['beta_dep']:+.3f} SD (p={h['p_dep']:.4f}). "
    if 'diff_ci' in h:
        lo, hi = h['diff_ci']
        line += verdict(lo > 0, f'SUPPORTED — deprivation effect larger than income (CI [{lo:.2f},{hi:.2f}]).',
                        f'NOT SUPPORTED at 95% — difference CI [{lo:.2f},{hi:.2f}] includes/below 0.')
    else:
        line += verdict(h['p_dep'] < 0.05 and h['beta_dep'] < 0,
                        'Deprivation significantly lowers the target; income comparison not possible (no income data).',
                        'No significant negative deprivation effect; income comparison not possible.')
    print(line)
h2 = RESULTS['H2']
print(f"H2: beta(retained | 2016 turnout, deprivation, metro) = {h2['beta_retained']:+.3f} SD (p={h2['p_retained']:.4f}); "
      f"retained wards: mean d_turnout {h2['mean_dturnout_retained']:+.3f} vs mean d_incumbent_share "
      f"{h2['mean_dshare_retained']:+.3f}.")
print('    ' + verdict(h2['beta_retained'] < 0 and h2['p_retained'] < 0.05 and
                       abs(h2['mean_dturnout_retained']) > abs(h2['mean_dshare_retained']),
                       'SUPPORTED — retention linked to lower turnout, and turnout moved more than incumbent share.',
                       'NOT (fully) SUPPORTED — see which condition failed above.'))
print(f"Model: {RESULTS['chosen_model']} chosen by one-SE rule (lowest MAE: {RESULTS['best_mae_model']}); "
      f"spatial-CV MAE {RESULTS['chosen_mae_pp']:.2f} pp vs baseline {RESULTS['baseline_mae_pp']:.2f} pp; R2 {RESULTS['chosen_r2']:.3f}.")
print(f"Classification: best classifier AUC {RESULTS['cls_best_auc']:.3f} vs regression-as-ranker AUC "
      f"{RESULTS['reg_as_ranker_auc']:.3f} -> " +
      verdict(RESULTS['reg_as_ranker_auc'] >= RESULTS['cls_best_auc'] - 0.02,
              'one regression model is sufficient for both forecasting and risk flagging.',
              'a dedicated classifier adds value for the risk list.'))
cov_txt = 'n/a' if RESULTS['recon_coverage'] is None else f"{RESULTS['recon_coverage']:.0%}"
print(f"Deprivation index: alpha={RESULTS['cronbach_alpha']:.2f}, PC1={RESULTS['pca_var_explained']:.0%}; "
      f"max VIF={RESULTS['max_vif']:.1f}; 2016->2021 reconciliation: {RESULTS['recon_method']}, coverage {cov_txt}.")

### 15.3 Limitations to state in the submission
- **Youth turnout is not observed at ward level**; youth conclusions rest on registration measures (T2) and on youth population share as a covariate.
- **Ecological inference:** ward patterns ≠ individual behaviour.
- **Two elections only** → "prolonged tenure" is a coarse proxy; add 2011/2006 via the voting-district crosswalk for true tenure length.
- **Census timing & income:** Census 2022 sits between elections; verify ward-level income availability before claiming the H1 income comparison.
- **Small n (≈242):** effect estimates have wide intervals; we report CIs and use conservative models.
- **Spatial dependence:** blocks reduce but do not remove it; a ward centroid or boundary file enables proper spatial clusters (and Moran's I on residuals).
- **Forecast level vs rank:** a national swing in 2026 moves all wards; the relative ranking is the robust output.